# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m01_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


실행 환경 준비 완료 (Python 3.14.10)


---

## 🎯 미션 01. 드론 조종 시뮬레이터 비행로그(스틱 조작, 자세각, 고도 유지, 경로 오차) 데이터 스키마 정의
**소요시간**: 약 1.5시간 | **필수 산출물**: data/schema/trn01_input_schema.json (필수), docs/trn01_data_spec.md (필수)

### 📋 미션 개요 (Overview)
드론 시뮬레이터 비행로그는 조종자 메타데이터, 전술 환경 변수, 시계열 조작 기록이 함께 들어오기 때문에, 분석 전에 잘못된 형식이나 보안 위반 데이터를 걸러내는 기준이 필수적이다. 본 미션에서는 가명 조종자 ID(P001~), 비행 조건·풍속·GPS 오차 등 전술 파라미터, 스틱·자세·고도·경로 레코드의 스키마와 허용 범위를 정의하고 자동 검증기를 구현한다. 정상 세션 20건과 오류 주입 5종을 모두 판별한 뒤 데이터 스키마(`trn01_input_schema.json`)와 명세서(`trn01_data_spec.md`)로 정리한다.

### 🎓 학습 목표 (Objectives)
시계열 비행로그 분석 전 단계에서 **데이터 무결성과 보안 규정을 코드 수준으로 강제하는 스키마 설계 및 유효성 검증 역량**을 익힌다. 세션 메타데이터(가명 ID, 코스, 기록 주기), 전술 파라미터(비행 조건, 풍속, GPS 오차, 영상 지연), 스틱·자세·고도·경로 레코드의 자료형과 허용 범위를 JSON 스키마로 정의하는 방법을 습득하고, 정상 세션 20건과 5종 오류(실명형 ID·미허용 조건·누락·범위 초과·시각 중복) 주입 테스트를 통해 경계값 검증과 명세 문서화 기준을 체득한다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션1 코드를 실행해 정상 세션 20건의 검증 결과를 확인한다.
#### 이 단계의 역할
미션1 코드를 실행해 정상 세션 20건의 검증 결과를 확인한다.

#### 해야 할 일

- 미션1 코드를 실행해 정상 세션 20건의 검증 결과를 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 정상 세션 20건 오류 0건, 주입 오류 5종 모두 검출 조종자 ID는 가명 형식(P001~)만 허용

#### 학습자가 제출해야 하는 내용

- 미션1 코드를 실행해 정상 세션 20건의 검증 결과를 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> JSON Schema(Draft-07)로 드론 비행로그의 session, tactical, records 구조를 검증하는 Python 함수를 작성해줘. pilot_id는 'P001' 형식의 가명 정규식만 허용하고, 실제 성명/군번 패턴은 차단해야 해.
**AI 응답 검증 포인트**

- 정상 세션 20건 실행 시 오탐(정상 로그를 오류로 분류) 없이 오류 0건으로 통과하는지 확인
- 생성된 스키마에 전술 파라미터(풍속, GPS 오차, 영상 지연) 범위 제약이 누락되지 않았는지 대조

#### Step 2. 오류 주입 결과 5건이 어떤 규칙에 걸렸는지 표로 정리한다.
#### 이 단계의 역할
오류 주입 결과 5건이 어떤 규칙에 걸렸는지 표로 정리한다.

#### 해야 할 일

- 오류 주입 결과 5건이 어떤 규칙에 걸렸는지 표로 정리한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 정상 세션 20건 오류 0건, 주입 오류 5종 모두 검출 조종자 ID는 가명 형식(P001~)만 허용

#### 학습자가 제출해야 하는 내용

- 오류 주입 결과 5건이 어떤 규칙에 걸렸는지 표로 정리한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 아래 5가지 오류 유형(실명형 ID, 미허용 비행조건, 필수 필드 누락, 스틱 범위 초과, 시각 중복)을 테스트 데이터에 주입했을 때, 어떤 스키마 규칙에서 걸리는지 표 형태로 정리하는 검증 스크립트를 짜줘.
**AI 응답 검증 포인트**

- 5종 오류가 각각 의도한 검증 규칙(정규식/enum/required/min-max/unique)에 정확히 매핑되는지 확인
- 첫 번째 오류에서 바로 중단되지 않고 모든 위반 항목이 누락 없이 리포트되는지 점검

#### Step 3. 데이터 명세 요약표에서 전술 파라미터(조건·풍속·GPS 오차·영상 지연)가 포함됐는지 확인...
#### 이 단계의 역할
데이터 명세 요약표에서 전술 파라미터(조건·풍속·GPS 오차·영상 지연)가 포함됐는지 확인한다.

#### 해야 할 일

- 데이터 명세 요약표에서 전술 파라미터(조건·풍속·GPS 오차·영상 지연)가 포함됐는지 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 정상 세션 20건 오류 0건, 주입 오류 5종 모두 검출 조종자 ID는 가명 형식(P001~)만 허용

#### 학습자가 제출해야 하는 내용

- 데이터 명세 요약표에서 전술 파라미터(조건·풍속·GPS 오차·영상 지연)가 포함됐는지 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> trn01_input_schema.json의 필드 정의를 바탕으로 개발자·교관이 함께 볼 수 있는 마크다운 데이터 명세 요약표(필드명, 타입, 허용 범위, 전술적 의미)를 자동 생성해줘.
**AI 응답 검증 포인트**

- 명세표의 허용 범위 수치가 실제 JSON 스키마 코드의 상수·제약 조건과 100% 일치하는지 교차 확인
- 보안 주의사항(가명 ID 사용 원칙, 실제 부대 정보 입력 금지)이 문서에 명시됐는지 점검

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
SCHEMA의 tactical 그룹에 시정(visibility_m, 0~10000) 필드를 추가하고 범위를 벗어난 값을 넣어 검출되는지 확인한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> tactical 그룹에 시정(visibility_m, 0~10000) 필드를 추가할 때, 기존 정상 세션 데이터와의 하위 호환성을 깨지 않으면서 범위 이탈(-1 또는 15000)을 검출하는 스키마 수정안을 제시해줘.
**AI 응답 검증 포인트**

- visibility_m 경계값(0, 10000)은 정상 통과하고 범위 이탈 값만 정확히 검출되는지 단위 테스트로 확인
- 기존 20건 정상 세션 검증 결과가 여전히 오류 0건인지 회귀 확인


In [2]:
"""드론 조종 시뮬레이터 비행로그(스틱 조작, 자세각, 고도, 위치·목표 경로)와 비행 세션 메타데이터(가명 조종자, 비행 조건, 전술 파라미터)의 스키마를 정의하고 가상 샘플 데이터를 검증한다."""
import json
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 드론 조종 시뮬레이터 비행로그 생성기 (실제 훈련생·장비 정보 없음, 조종자는 가명 P001~)
# 비행 1회 = 120초(10Hz): 경로 비행(0~60초) -> 호버링(60~85초) -> 돌발 장애물 회피(85초) -> 비상 착륙(100초 지시)
# 조종자마다 숨은 숙련도(skill)와 조작 습관(급조작 jerky, 과보정 overcorr)이 있고, 교관 점수는 이를 반영해 만든다.
# ---------------------------------------------------------------------------
DT, STEPS = 0.1, 1200
T = np.arange(STEPS) * DT
COND = {"day": dict(wind=0.4, gust=0.0, gps=0.25, delay=0.0), "night": dict(wind=0.4, gust=0.0, gps=0.5, delay=0.15),
        "gust": dict(wind=1.2, gust=2.5, gps=0.25, delay=0.0)}
OBST_T, LAND_T = 85.0, 100.0


def make_pilots(n=40, seed=7):
    rng = np.random.default_rng(seed)
    skill = rng.beta(2.2, 2.0, n)
    return pd.DataFrame({"pilot": [f"P{i + 1:03d}" for i in range(n)], "skill": skill,
                         "jerky": rng.random(n) < 0.15 + 0.35 * (1 - skill), "overcorr": rng.random(n) < 0.15 + 0.35 * (1 - skill)})


def target_path(t):
    """사각형 경로(40x30m, 고도 20m)를 3m/s로 비행 -> 60초 지점에서 호버링 -> 착륙 지시 후 고도 0."""
    s = np.minimum(t, 60.0) * 3.0 % 140.0
    x = np.select([s < 40, s < 70, s < 110], [s, 40.0, 40 - (s - 70)], 0.0)
    y = np.select([s < 40, s < 70, s < 110], [0.0, s - 40, 30.0], 30 - (s - 110))
    z = np.where(t >= LAND_T, 0.0, 20.0)
    return np.stack([x, y, z], -1)


def fly(pilots, cond="day", seed=0, delay_extra=0.0, obstacle_side=None):
    """여러 조종자의 비행 1회를 동시에 모의한다(벡터화). 반환: 로그 배열 dict, 비행별 정답 정보 DataFrame."""
    rng = np.random.default_rng(seed)
    c = COND[cond]
    delay_extra = delay_extra + c["delay"]
    n = len(pilots)
    sk, jk, oc = pilots["skill"].values, pilots["jerky"].values, pilots["overcorr"].values
    gain = (0.5 + 1.2 * sk) * np.where(oc, 1.5, 1.0)
    damp = np.where(oc, 0.25, 0.6 + 0.3 * sk)
    noise = 0.03 + 0.35 * (1 - sk)
    smooth = np.where(jk, 0.25, 0.75)
    delay = np.round((0.25 + 0.35 * (1 - sk) + delay_extra) / DT).astype(int)
    rt_obs = np.clip(0.5 + 1.5 * (1 - sk) + rng.normal(0, 0.2, n) + delay_extra, 0.3, 3.5)
    h_flare = np.clip(0.8 + 5.0 * sk - 1.5 * oc + rng.normal(0, 0.4, n), 0.3, None)
    side = rng.choice([-1, 1], n) if obstacle_side is None else np.full(n, obstacle_side)
    pos = np.tile(target_path(np.zeros(1))[0], (n, 1)).astype(float); vel = np.zeros((n, 3)); stick = np.zeros((n, 3))
    hist = np.zeros((STEPS, n, 3)); P = np.zeros((STEPS, n, 3)); S = np.zeros((STEPS, n, 3)); ATT = np.zeros((STEPS, n, 2))
    tgt_all = np.zeros((STEPS, n, 3)); vz_touch = np.full(n, np.nan); landed = np.zeros(n, bool)
    vmax = np.array([5.0, 5.0, 3.0])
    gust = np.zeros((n, 2))
    for i, t in enumerate(T):
        tgt = np.tile(target_path(np.array([t]))[0], (n, 1))
        if t >= OBST_T:                                              # 장애물: 6m 옆으로 회피해야 함(조종자는 반응시간 이후 인지)
            tgt_all[i] = tgt + np.c_[np.zeros(n), side * 6.0, np.zeros(n)]
            tgt[:, 1] += np.where(t >= OBST_T + rt_obs, side * 6.0, 0.0)
        else:
            tgt_all[i] = tgt
        hist[i] = pos
        seen = hist[np.maximum(i - delay, 0), np.arange(n)]          # 반응 지연만큼 늦게 본 위치
        cmd = np.clip(gain[:, None] * (tgt - seen) / 4.0 - damp[:, None] * vel / vmax + rng.normal(0, 1, (n, 3)) * noise[:, None], -1, 1)
        if t >= LAND_T:                                              # 비상 착륙: 하강하다가 h_flare 높이에서 감속(숙련자일수록 일찍 감속)
            cmd[:, 2] = np.where(seen[:, 2] > h_flare, -1.0, -0.12 + rng.normal(0, 0.05, n))
        spikes = (jk[:, None] & (rng.random((n, 3)) < 0.02)) * rng.choice([-1, 1], (n, 3)) * 0.9
        stick = np.clip(smooth[:, None] * stick + (1 - smooth[:, None]) * cmd + spikes, -1, 1)
        gust = 0.95 * gust + rng.normal(0, c["gust"] * 0.3, (n, 2))
        wind = np.c_[np.full(n, c["wind"]) + gust[:, 0], gust[:, 1], np.zeros(n)]
        vel += (stick * vmax - vel) * DT / 0.6
        vel[landed] = 0.0
        pos = pos + (vel + wind * (~landed)[:, None]) * DT
        touch = (pos[:, 2] <= 0) & ~landed & (t >= LAND_T)
        vz_touch[touch] = -vel[touch, 2]; landed |= touch
        pos[:, 2] = np.maximum(pos[:, 2], 0.0)
        P[i], S[i] = pos, stick
        ATT[i] = np.c_[stick[:, 1] * 28 + gust[:, 1] * 1.5, -stick[:, 0] * 28 + gust[:, 0] * 1.5]
    g = rng.normal(0, c["gps"], P.shape)
    log = {"t": T, "x": P[..., 0] + g[..., 0], "y": P[..., 1] + g[..., 1], "alt": P[..., 2] + g[..., 2] * 0.5,
           "stick_pitch": S[..., 0], "stick_roll": S[..., 1], "throttle": S[..., 2], "roll_deg": ATT[..., 0], "pitch_deg": ATT[..., 1],
           "tx": tgt_all[..., 0], "ty": tgt_all[..., 1], "talt": tgt_all[..., 2], "true_x": P[..., 0], "true_y": P[..., 1], "true_alt": P[..., 2]}
    k_obs = int((OBST_T + 2.5) / DT)
    lateral = np.abs(P[k_obs, :, 1] - P[int(OBST_T / DT), :, 1])
    truth = pd.DataFrame({"pilot": pilots["pilot"].values, "cond": cond, "skill": sk, "jerky": jk, "overcorr": oc, "rt_obs": rt_obs,
                          "near_collision": lateral < 2.5, "vz_touch": np.round(vz_touch, 2), "hard_landing": np.nan_to_num(vz_touch, nan=3.0) > 1.6,
                          "attitude_exceed": (np.abs(ATT) > 35).any(axis=2).sum(axis=0) >= 10})
    risk = truth[["near_collision", "hard_landing", "attitude_exceed"]].sum(axis=1)
    inst = 40 + 50 * sk - 5 * jk - 5 * oc - 4 * risk + rng.normal(0, 3, n)      # 교관 평가 점수(가상)
    truth["instructor"] = np.clip(inst, 0, 100).round(1)
    truth["grade"] = np.where(truth["instructor"] >= 75, "A", np.where(truth["instructor"] >= 55, "B", "C"))
    return log, truth


def one(log, j):
    """여러 조종자 로그에서 j번째 조종자 1명의 로그(DataFrame)를 꺼낸다."""
    return pd.DataFrame({k: (v if v.ndim == 1 else v[:, j]) for k, v in log.items()})
# ---------------------------------------------------------------------------

SCHEMA = {
    "session": {
        "session_id": {"type": str, "pattern": r"^S\d{4}$"},
        "pilot": {"type": str, "pattern": r"^P\d{3}$"},                       # 가명 ID만 허용(실명·군번 금지)
        "sim_course": {"type": str, "enum": ["RECT40x30_HOVER_OBS_LAND"]},
        "rate_hz": {"type": int, "enum": [10, 20, 50]},
        "duration_s": {"type": (int, float), "min": 10, "max": 1800},
    },
    "tactical": {
        "condition": {"type": str, "enum": list(COND)},
        "wind_mps": {"type": (int, float), "min": 0, "max": 20},
        "gps_error_m": {"type": (int, float), "min": 0, "max": 50},
        "video_delay_ms": {"type": (int, float), "min": 0, "max": 2000},
    },
    "record": {
        "t": {"type": (int, float), "min": 0},
        "stick_roll": {"type": (int, float), "min": -1, "max": 1},
        "stick_pitch": {"type": (int, float), "min": -1, "max": 1},
        "throttle": {"type": (int, float), "min": -1, "max": 1},
        "roll_deg": {"type": (int, float), "min": -90, "max": 90},
        "pitch_deg": {"type": (int, float), "min": -90, "max": 90},
        "alt": {"type": (int, float), "min": -1, "max": 500},
        "x": {"type": (int, float)}, "y": {"type": (int, float)},
        "tx": {"type": (int, float)}, "ty": {"type": (int, float)}, "talt": {"type": (int, float), "min": 0, "max": 500},
    },
}


def check(name, value, rule):
    if not isinstance(value, rule["type"]) or isinstance(value, bool):
        return [f"{name}: 자료형 오류({type(value).__name__})"]
    errs = []
    if "enum" in rule and value not in rule["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{value}'")
    if "min" in rule and value < rule["min"]:
        errs.append(f"{name}: 최소값 {rule['min']} 미만")
    if "max" in rule and value > rule["max"]:
        errs.append(f"{name}: 최대값 {rule['max']} 초과")
    if "pattern" in rule and not re.match(rule["pattern"], value):
        errs.append(f"{name}: 형식 불일치 '{value}'")
    return errs


def validate(sess):
    errs = []
    for g in ("session", "tactical"):
        for k, rule in SCHEMA[g].items():
            errs += [f"{g}.{k}: 필수 필드 누락"] if k not in sess.get(g, {}) else check(f"{g}.{k}", sess[g][k], rule)
    prev = -1.0
    for i, rec in enumerate(sess.get("records", [])):
        for k, rule in SCHEMA["record"].items():
            errs += [f"records[{i}].{k}: 필수 필드 누락"] if k not in rec else check(f"records[{i}].{k}", rec[k], rule)
        if isinstance(rec.get("t"), (int, float)) and rec["t"] <= prev:
            errs.append(f"records[{i}].t: 시각 역행/중복")
        prev = rec.get("t", prev)
    return errs


# [교육용가상자료] 조종자 20명 x 비행 1회 -> 세션 20건(각 세션에서 레코드 30개만 표본 검증)
pilots = make_pilots(20, seed=7)
log, truth = fly(pilots, "day", seed=1)
rng = np.random.default_rng(2)
sessions = []
for j in range(20):
    d = one(log, j).iloc[::40][list(SCHEMA["record"])].round(3)
    sessions.append({"session": {"session_id": f"S{j + 1:04d}", "pilot": pilots["pilot"][j], "sim_course": "RECT40x30_HOVER_OBS_LAND",
                                 "rate_hz": 10, "duration_s": 120.0},
                     "tactical": {"condition": "day", "wind_mps": float(round(rng.uniform(0, 3), 1)), "gps_error_m": 0.25,
                                  "video_delay_ms": int(rng.integers(50, 250))},
                     "records": [{k: float(v) for k, v in r.items()} for r in d.to_dict("records")]})
ok = sum(1 for s in sessions if not validate(s))
print(f"정상 세션 {len(sessions)}건(레코드 {sum(len(s['records']) for s in sessions)}개) 중 검증 통과 {ok}건, 오류 {len(sessions) - ok}건")

bad = json.loads(json.dumps(sessions[0]))
bad["session"]["pilot"] = "KIM-01"          # 실명형 ID 입력
bad["tactical"]["condition"] = "storm"
del bad["tactical"]["video_delay_ms"]
bad["records"][3]["stick_roll"] = 1.8
bad["records"][5]["t"] = bad["records"][4]["t"]
print("[오류 주입 결과]")
for e in validate(bad):
    print(" -", e)

rows = []
for g in SCHEMA:
    for k, r in SCHEMA[g].items():
        t = r["type"].__name__ if isinstance(r["type"], type) else "number"
        rows.append((g, k, t, str({x: y for x, y in r.items() if x != "type"})[:45]))
print("\n[데이터 명세 요약]")
print(pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙"]).to_string(index=False))
print("\n비행 단계: 경로 0~60초 / 호버링 60~85초 / 장애물 회피 85초 / 비상 착륙 지시 100초, 10Hz 기록")
print("개인정보: 조종자는 가명 ID(P001~)로만 저장하고, 실명·군번·소속은 별도 대장에서 교관만 관리한다.")
# [산출물 파일 저장]
import os, json
os.makedirs("data/schema", exist_ok=True)
os.makedirs("docs", exist_ok=True)
with open("data/schema/trn01_input_schema.json", "w", encoding="utf-8") as _f:
    json.dump(SCHEMA, _f, ensure_ascii=False, indent=2, default=lambda x: x.__name__ if isinstance(x, type) else str(x))
_spec_df = pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙"])
with open("docs/trn01_data_spec.md", "w", encoding="utf-8") as _f:
    _f.write("# TRN01 비행로그 데이터 명세서\n\n```\n" + _spec_df.to_string(index=False) + "\n```\n")
print("[산출물 저장 완료] data/schema/trn01_input_schema.json, docs/trn01_data_spec.md")


정상 세션 20건(레코드 600개) 중 검증 통과 20건, 오류 0건
[오류 주입 결과]
 - session.pilot: 형식 불일치 'KIM-01'
 - tactical.condition: 허용되지 않은 값 'storm'
 - tactical.video_delay_ms: 필수 필드 누락
 - records[3].stick_roll: 최대값 1 초과
 - records[5].t: 시각 역행/중복

[데이터 명세 요약]
      그룹             필드    자료형                                     규칙
 session     session_id    str               {'pattern': '^S\\d{4}$'}
 session          pilot    str               {'pattern': '^P\\d{3}$'}
 session     sim_course    str {'enum': ['RECT40x30_HOVER_OBS_LAND']}
 session        rate_hz    int                 {'enum': [10, 20, 50]}
 session     duration_s number               {'min': 10, 'max': 1800}
tactical      condition    str     {'enum': ['day', 'night', 'gust']}
tactical       wind_mps number                  {'min': 0, 'max': 20}
tactical    gps_error_m number                  {'min': 0, 'max': 50}
tactical video_delay_ms number                {'min': 0, 'max': 2000}
  record              t number                             {'min

<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
정상 세션 20건(레코드 600개) 중 검증 통과 20건, 오류 0건
[오류 주입 결과]
 - session.pilot: 형식 불일치 'KIM-01'
 - tactical.condition: 허용되지 않은 값 'storm'
 - tactical.video_delay_ms: 필수 필드 누락
 - records[3].stick_roll: 최대값 1 초과
 - records[5].t: 시각 역행/중복

[데이터 명세 요약]
      그룹             필드    자료형                                     규칙
 session     session_id    str               {'pattern': '^S\\d{4}$'}
 session          pilot    str               {'pattern': '^P\\d{3}$'}
 session     sim_course    str {'enum': ['RECT40x30_HOVER_OBS_LAND']}
 session        rate_hz    int                 {'enum': [10, 20, 50]}
 session     duration_s number               {'min': 10, 'max': 1800}
tactical      condition    str     {'enum': ['day', 'night', 'gust']}
tactical       wind_mps number                  {'min': 0, 'max': 20}
tactical    gps_error_m number                  {'min': 0, 'max': 50}
tactical video_delay_ms number                {'min': 0, 'max': 2000}
  record              t number                             {'min': 0}
  record     stick_roll number                  {'min': -1, 'max': 1}
  record    stick_pitch number                  {'min': -1, 'max': 1}
  record       throttle number                  {'min': -1, 'max': 1}
  record       roll_deg number                {'min': -90, 'max': 90}
  record      pitch_deg number                {'min': -90, 'max': 90}
  record            alt number                {'min': -1, 'max': 500}
  record              x number                                     {}
  record              y number                                     {}
  record             tx number                                     {}
  record             ty number                                     {}
  record           talt number                 {'min': 0, 'max': 500}

비행 단계: 경로 0~60초 / 호버링 60~85초 / 장애물 회피 85초 / 비상 착륙 지시 100초, 10Hz 기록
개인정보: 조종자는 가명 ID(P001~)로만 저장하고, 실명·군번·소속은 별도 대장에서 교관만 관리한다.
```
</details>
